# 🔍 On-Chain Detective Framework

**Быстрый старт** — полное расследование за 5 минут.

1. Убедись, что в `.env` прописаны ключи: `ETHERSCAN_API_KEY`, `OPENAI_API_KEY`, `OPENAI_BASE_URL`
2. Запускай ячейки по порядку

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\23ale\onchain_detective")

from dotenv import load_dotenv
from pathlib import Path
import os

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(root / ".env")

print(f"✅ .env загружен из {root}")

from onchain_detective.config import settings
from onchain_detective.providers.etherscan import EtherscanProvider
from onchain_detective.core.cluster import WalletClusterer
from onchain_detective.core.insider import InsiderDetector
from onchain_detective.core.whale import WhaleMonitor
from onchain_detective.ai.openai_analyzer import AIAnalyzer

print(f"✅ Сеть: {settings.NETWORK}")
print(f"✅ Все модули загружены")

In [ ]:
# Инициализация провайдера
provider = EtherscanProvider(settings.ETHERSCAN_API_KEY)
print("✅ Провайдер готов")

## 1. Кластеризация кошельков

Введи адрес для анализа:

In [ ]:
# ВВЕДИ АДРЕС ДЛЯ АНАЛИЗА
TARGET = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"  # ← Замени на свой

clusterer = WalletClusterer(provider)
graph = clusterer.analyze_wallet(TARGET, depth=1, max_wallets=15)
cluster_report = clusterer.export_report()

print(f"Узлов: {cluster_report['total_wallets_analyzed']}")
print(f"Связей: {cluster_report['total_connections']}")
print(f"Подозрительных кластеров: {cluster_report['suspicious_clusters']}")

# Граф
graph.render("cluster_graph.png")
from IPython.display import Image
Image("cluster_graph.png")

## 2. Детектор инсайдеров

Введи адрес токена для поиска инсайдеров:

In [ ]:
# ВВЕДИ АДРЕС ТОКЕНА
TOKEN = "0x6982508145454Ce325dDbE47a25d4ec3d2311933"  # ← Замени на свой

detector = InsiderDetector(provider)
insider_report = detector.analyze_token(TOKEN, check_top_buyers=5)

print(f"Ранних покупателей: {insider_report['total_early_buyers']}")
print(f"Подозрительных: {insider_report['suspicious_count']}")
print(f"Уровень риска: {insider_report['risk_level']}")

## 3. AI-расследование

AI анализирует все данные и выдаёт вердикт:

In [ ]:
ai = AIAnalyzer(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://api.routerai.ru/api/v1"),
    model=settings.AI_MODEL,
)

verdict = ai.investigate(cluster_report, insider_report)

print(f"🔴 Уровень риска: {verdict.get('risk_level')}")
print(f"📝 Резюме: {verdict.get('summary')}")
print(f"📋 Вердикт: {verdict.get('verdict')}")
print(f"💡 Рекомендация: {verdict.get('recommendation')}")

if verdict.get('red_flags'):
    print("\n🚩 Красные флаги:")
    for flag in verdict['red_flags']:
        print(f"  • {flag}")

## 4. Монитор китов (опционально)

Для работы нужен Telegram-бот. Настрой `TELEGRAM_BOT_TOKEN` и `TELEGRAM_CHAT_ID` в `.env`.

In [ ]:
monitor = WhaleMonitor(provider)
whales = monitor.load_top_whales(5)

print(f"Отслеживается китов: {len(whales)}")
for w in whales:
    print(f"  {w[:10]}...")

# Однократная проверка
alerts = monitor.scan_and_alert()
print(f"Найдено движений: {len(alerts)}")